# Assignment 2: crime / violence classification

Decision the model supports: flag **1 fps samples** from city CCTV that look like
`violence` or `weaponized` activity (labels as defined by the Roboflow dataset),
versus `normal`. Deployment stays image-level (no video model). Who acts, what each
error costs, thresholds, and whether to ship are graded decisions and stay in
REPORT.md — this notebook produces the numbers those decisions need.

**Dataset:** [`pdl-qtfdi/crime-detection-jekv6`](https://universe.roboflow.com/pdl-qtfdi/crime-detection-jekv6) version `1`
(image classification, three classes). We use the public export as labelled; we do
**not** fork it on Universe.

**Pipeline (built step by step):**
1. Download the Roboflow export and rebuild splits **by source video** (this step).
2. Inspect the data.
3. Fine-tune YOLO11 classification, compare baselines, cost the confusion matrix,
   save weights and artifacts for the Gradio app.

Run end-to-end with `uv run python main.py train` once the later sections exist.
Style follows the course card-detector notebook: short code, prose that says why.


In [ ]:
from __future__ import annotations

import random
import re
import shutil
from collections import Counter, defaultdict
from pathlib import Path

import torch

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Colab ships its own GPU PyTorch — do not uv sync here (see SETUP.md).
    import subprocess
    import sys

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "ultralytics", "roboflow", "pillow"]
    )

ROOT = Path.cwd()
if not (ROOT / "api_key.py").is_file():
    raise SystemExit("Run this notebook from the assignment repo root.")

DEVICE = 0 if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
SEED = 0

ROBOFLOW_WORKSPACE = "pdl-qtfdi"
ROBOFLOW_PROJECT = "crime-detection-jekv6"
ROBOFLOW_VERSION = 1

RAW_DIR = ROOT / "datasets" / "crime-detection-jekv6"
SPLIT_DIR = ROOT / "datasets" / "crime-detection-jekv6-videosplit"
CLASSES = ("normal", "violence", "weaponized")
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
VIDEO_ID_RE = re.compile(r"^([nvw]\d+)_", re.IGNORECASE)

# Fractions of *videos per class* (confirmed): 70 / 15 / 15
TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.70, 0.15, 0.15

WEIGHTS_DIR = ROOT / "weights"
ARTIFACTS_DIR = ROOT / "artifacts"

# Locked training defaults (Step 3)
MODEL_NAME = "yolo11n-cls.pt"
EPOCHS = 15
IMGSZ = 224
BATCH = 16

print(f"colab: {IN_COLAB}")
print(f"device: {DEVICE}")
print(f"raw export: {RAW_DIR}")
print(f"video-level split: {SPLIT_DIR}")
print(f"weights -> {WEIGHTS_DIR}")
print(f"train: {MODEL_NAME} epochs={EPOCHS} imgsz={IMGSZ} batch={BATCH}")


In [ ]:
from api_key import load_api_key
from roboflow import Roboflow


def ensure_roboflow_export(destination: Path) -> Path:
    """Download version 1 once; reuse the folder on later runs."""
    marker = destination / "train"
    if marker.is_dir() and any(marker.iterdir()):
        print(f"already on disk: {destination}")
        return destination

    destination.parent.mkdir(parents=True, exist_ok=True)
    rf = Roboflow(api_key=load_api_key())
    project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
    print(f"downloading {ROBOFLOW_WORKSPACE}/{ROBOFLOW_PROJECT} v{ROBOFLOW_VERSION} …")
    project.version(ROBOFLOW_VERSION).download("folder", location=str(destination))
    return destination


ensure_roboflow_export(RAW_DIR)


In [ ]:
def video_id_from_name(path: Path) -> str | None:
    match = VIDEO_ID_RE.match(path.name)
    return match.group(1).lower() if match else None


def collect_videos(raw_dir: Path) -> dict[str, dict]:
    """Map video_id -> class_name and file list."""
    videos: dict[str, dict] = {}
    for split in ("train", "valid", "test"):
        split_dir = raw_dir / split
        if not split_dir.is_dir():
            continue
        for class_dir in sorted(p for p in split_dir.iterdir() if p.is_dir()):
            class_name = class_dir.name
            for image in class_dir.rglob("*"):
                if image.suffix.lower() not in IMAGE_EXTS:
                    continue
                vid = video_id_from_name(image)
                if vid is None:
                    raise ValueError(f"cannot parse video id: {image.name}")
                entry = videos.setdefault(vid, {"class_name": class_name, "files": []})
                if entry["class_name"] != class_name:
                    raise ValueError(
                        f"video {vid} has conflicting classes "
                        f"{entry['class_name']} vs {class_name}"
                    )
                entry["files"].append(image)
    return videos


def allocate_counts(n: int) -> tuple[int, int, int]:
    """Integer counts that sum to n; leftovers go to train."""
    n_val = int(n * VAL_FRAC)
    n_test = int(n * TEST_FRAC)
    n_train = n - n_val - n_test
    return n_train, n_val, n_test


def assign_splits(videos: dict[str, dict], seed: int = SEED) -> dict[str, str]:
    """Return video_id -> train/valid/test, stratified by class."""
    by_class: dict[str, list[str]] = defaultdict(list)
    for vid, meta in videos.items():
        by_class[meta["class_name"]].append(vid)

    assignment: dict[str, str] = {}
    rng = random.Random(seed)
    for class_name in CLASSES:
        ids = sorted(by_class[class_name])
        rng.shuffle(ids)
        n_train, n_val, n_test = allocate_counts(len(ids))
        train_ids = ids[:n_train]
        val_ids = ids[n_train : n_train + n_val]
        test_ids = ids[n_train + n_val :]
        assert len(test_ids) == n_test
        for vid in train_ids:
            assignment[vid] = "train"
        for vid in val_ids:
            assignment[vid] = "valid"
        for vid in test_ids:
            assignment[vid] = "test"
        print(
            f"{class_name:12s} videos={len(ids):3d} -> "
            f"train {n_train}, valid {n_val}, test {n_test}"
        )
    return assignment


def write_videosplit(
    videos: dict[str, dict],
    assignment: dict[str, str],
    destination: Path,
) -> None:
    if destination.exists():
        shutil.rmtree(destination)
    for split in ("train", "valid", "test"):
        for class_name in CLASSES:
            (destination / split / class_name).mkdir(parents=True, exist_ok=True)

    for vid, meta in videos.items():
        split = assignment[vid]
        class_name = meta["class_name"]
        out_dir = destination / split / class_name
        for src in meta["files"]:
            shutil.copy2(src, out_dir / src.name)


def summarise_split(destination: Path) -> None:
    print("\nimage counts after video-level split:")
    grand = Counter()
    for split in ("train", "valid", "test"):
        counts = Counter()
        for class_name in CLASSES:
            folder = destination / split / class_name
            n = sum(1 for p in folder.iterdir() if p.suffix.lower() in IMAGE_EXTS)
            counts[class_name] = n
            grand[class_name] += n
        total = sum(counts.values())
        parts = ", ".join(f"{c}={counts[c]}" for c in CLASSES)
        print(f"  {split:5s} n={total:5d}  ({parts})")
    print(f"  ALL   n={sum(grand.values()):5d}")


def assert_no_video_leakage(destination: Path) -> None:
    seen: dict[str, str] = {}
    leaks = []
    for split in ("train", "valid", "test"):
        for class_dir in (destination / split).iterdir():
            for image in class_dir.iterdir():
                if image.suffix.lower() not in IMAGE_EXTS:
                    continue
                vid = video_id_from_name(image)
                if vid in seen and seen[vid] != split:
                    leaks.append((vid, seen[vid], split))
                else:
                    seen[vid] = split
    if leaks:
        raise AssertionError(f"video leakage remains, e.g. {leaks[:5]}")
    print(f"\nleakage check: OK ({len(seen)} videos, each in exactly one split)")


videos = collect_videos(RAW_DIR)
print(f"source videos: {len(videos)}")
print(f"frames total:  {sum(len(v['files']) for v in videos.values())}")
print()
assignment = assign_splits(videos)
write_videosplit(videos, assignment, SPLIT_DIR)
summarise_split(SPLIT_DIR)
assert_no_video_leakage(SPLIT_DIR)
print(f"\nwrote {SPLIT_DIR}")


In [ ]:
from PIL import Image, ImageDraw, ImageFont


def split_composition(split_dir: Path = SPLIT_DIR) -> None:
    """Print image and unique-video counts per split and class."""
    print(f"composition for {split_dir.name}\n")
    header = f"{'split':6} {'class':12} {'images':>7} {'videos':>7} {'img%':>6}"
    print(header)
    print("-" * len(header))
    for split in ("train", "valid", "test"):
        split_total = 0
        rows = []
        for class_name in CLASSES:
            folder = split_dir / split / class_name
            files = [p for p in folder.iterdir() if p.suffix.lower() in IMAGE_EXTS]
            vids = {video_id_from_name(p) for p in files}
            rows.append((class_name, len(files), len(vids)))
            split_total += len(files)
        for class_name, n_img, n_vid in rows:
            share = 100 * n_img / split_total if split_total else 0
            print(f"{split:6} {class_name:12} {n_img:7d} {n_vid:7d} {share:5.1f}%")
        print(f"{split:6} {'TOTAL':12} {split_total:7d}")
        print()


def class_sheet(
    class_name: str,
    split: str = "train",
    count: int = 6,
    columns: int = 3,
    thumb: int = 224,
    seed: int = SEED,
):
    """Contact sheet of random frames for one class (reproducible seed)."""
    folder = SPLIT_DIR / split / class_name
    files = sorted(p for p in folder.iterdir() if p.suffix.lower() in IMAGE_EXTS)
    rng = random.Random(seed)
    sample = files if len(files) <= count else rng.sample(files, count)
    rows = (len(sample) + columns - 1) // columns
    sheet = Image.new("RGB", (columns * thumb, rows * thumb), "black")
    try:
        font = ImageFont.load_default()
    except Exception:
        font = None
    for position, path in enumerate(sample):
        image = Image.open(path).convert("RGB").resize((thumb, thumb))
        draw = ImageDraw.Draw(image)
        label = video_id_from_name(path) or path.stem[:12]
        draw.rectangle((0, 0, thumb, 14), fill=(0, 0, 0))
        draw.text((2, 1), f"{label}", fill=(0, 255, 0), font=font)
        sheet.paste(image, ((position % columns) * thumb, (position // columns) * thumb))
    return sheet


split_composition()
print("sample sheets (train, seed=0):")
for class_name in CLASSES:
    sheet = class_sheet(class_name)
    display_path = ROOT / "runs" / "inspect" / f"{class_name}_train_sheet.jpg"
    display_path.parent.mkdir(parents=True, exist_ok=True)
    sheet.save(display_path)
    print(f"  saved {display_path.relative_to(ROOT)}")
    try:
        display(sheet)  # Jupyter
    except NameError:
        pass  # main.py train / plain script: files on disk are enough


In [ ]:
# When run via `main.py train` (nbconvert -> script), there is no IPython display().
# Re-show paths only; open the JPEGs under runs/inspect/ if you are not in a notebook UI.
inspect_dir = ROOT / "runs" / "inspect"
for class_name in CLASSES:
    path = inspect_dir / f"{class_name}_train_sheet.jpg"
    assert path.is_file(), path
    print(path.relative_to(ROOT), f"({path.stat().st_size} bytes)")


In [ ]:
import json

from ultralytics import YOLO

assert SPLIT_DIR.is_dir() and any((SPLIT_DIR / "train").iterdir()), (
    f"Missing video-level split at {SPLIT_DIR}. Run Sections 1–2 first."
)

WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"training on device={DEVICE} data={SPLIT_DIR}")
crime_model = YOLO(MODEL_NAME)

train_results = crime_model.train(
    data=str(SPLIT_DIR),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    project=str(ROOT / "runs"),
    name="crime_videosplit",
    exist_ok=True,
    seed=SEED,
    deterministic=False,
)

RUN_DIR = Path(crime_model.trainer.save_dir)
BEST_WEIGHTS = RUN_DIR / "weights" / "best.pt"
LAST_WEIGHTS = RUN_DIR / "weights" / "last.pt"
DEST = WEIGHTS_DIR / "best.pt"
shutil.copy2(BEST_WEIGHTS, DEST)

print()
print("run directory:", RUN_DIR)
print("best weights: ", BEST_WEIGHTS, f"({BEST_WEIGHTS.stat().st_size / 1e6:.1f} MB)")
print("copied to:    ", DEST)
print("classes:", YOLO(str(DEST)).names)

# Lightweight run record for REPORT.md / later app wiring (no cost decisions yet).
run_record = {
    "model": MODEL_NAME,
    "epochs": EPOCHS,
    "imgsz": IMGSZ,
    "batch": BATCH,
    "seed": SEED,
    "device": str(DEVICE),
    "data": str(SPLIT_DIR.relative_to(ROOT)) if SPLIT_DIR.is_relative_to(ROOT) else str(SPLIT_DIR),
    "roboflow": f"{ROBOFLOW_WORKSPACE}/{ROBOFLOW_PROJECT}/{ROBOFLOW_VERSION}",
    "split": "video-level 70/15/15 stratified by class",
    "weights": str(DEST.relative_to(ROOT)) if DEST.is_relative_to(ROOT) else str(DEST),
    "run_dir": str(RUN_DIR),
    "colab": bool(IN_COLAB),
}
(ARTIFACTS_DIR / "train_run.json").write_text(
    json.dumps(run_record, indent=2) + "\n", encoding="utf-8"
)
print("wrote", ARTIFACTS_DIR / "train_run.json")


In [ ]:
# Need json in this namespace when the train cell runs after a fresh kernel —
# import here so nbconvert order cannot bite us.
import json

results_png = RUN_DIR / "results.png"
if results_png.is_file():
    print("training curves:", results_png)
    try:
        from IPython.display import Image as IPyImage, display

        display(IPyImage(filename=str(results_png)))
    except Exception:
        pass
else:
    print("no results.png yet (unexpected for a finished Ultralytics run)")


In [ ]:
import json
from collections import Counter

import numpy as np
from ultralytics import YOLO

POSITIVE_CLASSES = ("violence", "weaponized")  # confirmed: both positive; separate FN costs later
NEGATIVE_CLASS = "normal"

WEIGHTS_PATH = WEIGHTS_DIR / "best.pt"
TEST_DIR = SPLIT_DIR / "test"
TRAIN_DIR = SPLIT_DIR / "train"


def list_split_images(split_dir: Path) -> list[tuple[Path, str]]:
    rows = []
    for class_name in CLASSES:
        folder = split_dir / class_name
        for path in sorted(p for p in folder.iterdir() if p.suffix.lower() in IMAGE_EXTS):
            rows.append((path, class_name))
    return rows


def is_positive(class_name: str) -> bool:
    return class_name in POSITIVE_CLASSES


def confusion_counts(y_true: list[str], y_pred: list[str], labels: tuple[str, ...]) -> list[list[int]]:
    index = {c: i for i, c in enumerate(labels)}
    matrix = [[0 for _ in labels] for _ in labels]
    for t, p in zip(y_true, y_pred):
        matrix[index[t]][index[p]] += 1
    return matrix


def per_class_prf(matrix: list[list[int]], labels: tuple[str, ...]) -> dict:
    out = {}
    for i, label in enumerate(labels):
        tp = matrix[i][i]
        fp = sum(matrix[r][i] for r in range(len(labels)) if r != i)
        fn = sum(matrix[i][c] for c in range(len(labels)) if c != i)
        prec = tp / (tp + fp) if (tp + fp) else 0.0
        rec = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
        support = sum(matrix[i])
        out[label] = {
            "precision": prec,
            "recall": rec,
            "f1": f1,
            "support": support,
            "tp": tp,
            "fp": fp,
            "fn": fn,
        }
    return out


def binary_counts(y_true: list[str], y_pred: list[str]) -> dict:
    tp = fp = tn = fn = 0
    for t, p in zip(y_true, y_pred):
        t_pos, p_pos = is_positive(t), is_positive(p)
        if t_pos and p_pos:
            tp += 1
        elif (not t_pos) and p_pos:
            fp += 1
        elif (not t_pos) and (not p_pos):
            tn += 1
        else:
            fn += 1
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    spec = tn / (tn + fp) if (tn + fp) else 0.0
    acc = (tp + tn) / max(len(y_true), 1)
    return {
        "tp": tp,
        "fp": fp,
        "tn": tn,
        "fn": fn,
        "precision": prec,
        "recall": rec,
        "specificity": spec,
        "accuracy": acc,
        "positive_definition": list(POSITIVE_CLASSES),
    }


def accuracy(y_true: list[str], y_pred: list[str]) -> float:
    return sum(t == p for t, p in zip(y_true, y_pred)) / max(len(y_true), 1)


assert WEIGHTS_PATH.is_file(), WEIGHTS_PATH
test_rows = list_split_images(TEST_DIR)
train_rows = list_split_images(TRAIN_DIR)
print(f"test images: {len(test_rows)}")
print(f"train images: {len(train_rows)}")

train_prior = Counter(c for _, c in train_rows)
majority_class = train_prior.most_common(1)[0][0]
print(f"train class counts: {dict(train_prior)}")
print(f"majority-class baseline predicts: {majority_class}")

# --- Finetuned model: collect softmax + argmax on test ---
model = YOLO(str(WEIGHTS_PATH))
name_to_idx = {v: int(k) for k, v in model.names.items()}
idx_to_name = {int(k): v for k, v in model.names.items()}
assert set(idx_to_name.values()) == set(CLASSES)

y_true: list[str] = []
y_pred: list[str] = []
records = []

paths = [p for p, _ in test_rows]
# Predict in chunks — a single call on ~1.8k paths OOMs on some machines.
pred_results = []
chunk_size = 64
for start in range(0, len(paths), chunk_size):
    chunk = paths[start : start + chunk_size]
    pred_results.extend(model.predict(chunk, imgsz=IMGSZ, verbose=False, stream=False))

for (path, true_label), result in zip(test_rows, pred_results):
    probs = result.probs.data.detach().cpu().numpy().astype(float)
    pred_idx = int(result.probs.top1)
    pred_label = idx_to_name[pred_idx]
    p_normal = float(probs[name_to_idx["normal"]])
    p_violence = float(probs[name_to_idx["violence"]])
    p_weaponized = float(probs[name_to_idx["weaponized"]])
    p_positive = p_violence + p_weaponized
    y_true.append(true_label)
    y_pred.append(pred_label)
    records.append(
        {
            "path": str(path.relative_to(ROOT)) if path.is_relative_to(ROOT) else str(path),
            "true": true_label,
            "pred": pred_label,
            "true_positive": is_positive(true_label),
            "pred_positive": is_positive(pred_label),
            "p_normal": p_normal,
            "p_violence": p_violence,
            "p_weaponized": p_weaponized,
            "p_positive": p_positive,
            "top1_conf": float(result.probs.top1conf),
        }
    )

cm = confusion_counts(y_true, y_pred, CLASSES)
prf = per_class_prf(cm, CLASSES)
bin_metrics = binary_counts(y_true, y_pred)
acc3 = accuracy(y_true, y_pred)

print("\n=== Finetuned yolo11n-cls (argmax) ===")
print(f"3-class accuracy: {acc3:.4f}")
print("confusion rows=true, cols=pred", list(CLASSES))
for row, label in zip(cm, CLASSES):
    print(f"  {label:12s} {row}")
print("per-class:")
for label, stats in prf.items():
    print(
        f"  {label:12s} P={stats['precision']:.3f} R={stats['recall']:.3f} "
        f"F1={stats['f1']:.3f} n={stats['support']}"
    )
print(
    "binary (pos=violence|weaponized): "
    f"P={bin_metrics['precision']:.3f} R={bin_metrics['recall']:.3f} "
    f"acc={bin_metrics['accuracy']:.3f} "
    f"TP={bin_metrics['tp']} FP={bin_metrics['fp']} "
    f"TN={bin_metrics['tn']} FN={bin_metrics['fn']}"
)

# --- Baseline 1: majority class (3-way) ---
y_maj = [majority_class] * len(y_true)
cm_maj = confusion_counts(y_true, y_maj, CLASSES)
bin_maj = binary_counts(y_true, y_maj)
acc_maj = accuracy(y_true, y_maj)

# --- Baseline 2: zero-shot stand-in = predict nothing (always normal) ---
# No label-compatible zero-shot exists: ImageNet yolo11n-cls and COCO detectors
# have no normal/violence/weaponized outputs, so they cannot raise our alerts.
y_none = [NEGATIVE_CLASS] * len(y_true)
cm_none = confusion_counts(y_true, y_none, CLASSES)
bin_none = binary_counts(y_true, y_none)
acc_none = accuracy(y_true, y_none)

print("\n=== Baselines (same test split) ===")
print(
    f"majority ({majority_class}): 3-cls acc={acc_maj:.4f} "
    f"binary P={bin_maj['precision']:.3f} R={bin_maj['recall']:.3f} acc={bin_maj['accuracy']:.3f}"
)
print(
    f"zero-shot stand-in / predict-nothing (always normal): 3-cls acc={acc_none:.4f} "
    f"binary P={bin_none['precision']:.3f} R={bin_none['recall']:.3f} "
    f"acc={bin_none['accuracy']:.3f}"
)

zeroshot_note = (
    "No label-compatible zero-shot: pretrained yolo11n-cls (ImageNet) and COCO "
    "detectors cannot emit normal/violence/weaponized. Zero-shot stand-in = "
    "predict-nothing (always normal)."
)
print("zero-shot note:", zeroshot_note)

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
eval_payload = {
    "positive_classes": list(POSITIVE_CLASSES),
    "negative_class": NEGATIVE_CLASS,
    "weights": str(WEIGHTS_PATH.relative_to(ROOT)),
    "test_dir": str(TEST_DIR.relative_to(ROOT)),
    "n_test": len(y_true),
    "finetuned": {
        "accuracy_3class": acc3,
        "confusion_matrix": {"labels": list(CLASSES), "matrix": cm},
        "per_class": prf,
        "binary": bin_metrics,
    },
    "baselines": {
        "majority_class": {
            "predicts": majority_class,
            "train_counts": dict(train_prior),
            "accuracy_3class": acc_maj,
            "confusion_matrix": {"labels": list(CLASSES), "matrix": cm_maj},
            "binary": bin_maj,
        },
        "predict_nothing": {
            "predicts": NEGATIVE_CLASS,
            "accuracy_3class": acc_none,
            "confusion_matrix": {"labels": list(CLASSES), "matrix": cm_none},
            "binary": bin_none,
            "zero_shot_interpretation": zeroshot_note,
        },
    },
    "notes": {
        "overfit": "Val top-1 peaked epoch 4 (~0.75); epoch 15 ~0.69. Using best.pt.",
        "top5": "Ignore top-5 with 3 classes; it is trivially 1.0.",
        "fn_costs": "violence vs weaponized FN costs will differ in Step 5; not applied yet.",
    },
}
(ARTIFACTS_DIR / "eval_test.json").write_text(
    json.dumps(eval_payload, indent=2) + "\n", encoding="utf-8"
)
(ARTIFACTS_DIR / "test_predictions.json").write_text(
    json.dumps({"records": records}, indent=2) + "\n", encoding="utf-8"
)
print("\nwrote", ARTIFACTS_DIR / "eval_test.json")
print("wrote", ARTIFACTS_DIR / "test_predictions.json", f"({len(records)} rows)")


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# --- Locked costs & production mix (your graded figures) ---
COST_FP = 5.0
COST_FN_VIOLENCE = 1000.0
COST_FN_WEAPONIZED = 100000.0
COST_TP = 0.0
COST_TN = 0.0

PROD_P_POSITIVE = 0.0001  # 0.01%
PROD_P_WEAPONIZED_GIVEN_POS = 0.20
PROD_P_VIOLENCE = PROD_P_POSITIVE * (1.0 - PROD_P_WEAPONIZED_GIVEN_POS)
PROD_P_WEAPONIZED = PROD_P_POSITIVE * PROD_P_WEAPONIZED_GIVEN_POS
PROD_P_NORMAL = 1.0 - PROD_P_POSITIVE

# Operating point: wide review band; t_alert filled from argmin after the sweep.
T_CLEAR = 0.01  # auto-clear only very low incident scores
T_ALERT = None  # filled from production-cost argmin
COST_REVIEW = 0.5  # 50 currency/h ÷ 100 images/h

PRED_PATH = ARTIFACTS_DIR / "test_predictions.json"
assert PRED_PATH.is_file(), PRED_PATH
records = json.loads(PRED_PATH.read_text(encoding="utf-8"))["records"]

y_true = np.array([r["true"] for r in records])
p_pos = np.array([r["p_positive"] for r in records], dtype=float)

is_normal = y_true == "normal"
is_violence = y_true == "violence"
is_weaponized = y_true == "weaponized"
n_normal = int(is_normal.sum())
n_violence = int(is_violence.sum())
n_weaponized = int(is_weaponized.sum())
print(f"loaded {len(records)} predictions  n={n_normal} v={n_violence} w={n_weaponized}")


def rates_at_threshold(threshold: float) -> dict:
    """Hard alert if p_positive >= threshold; else clear."""
    alert = p_pos >= threshold
    fpr = float(alert[is_normal].mean()) if n_normal else 0.0
    miss_v = float((~alert)[is_violence].mean()) if n_violence else 0.0
    miss_w = float((~alert)[is_weaponized].mean()) if n_weaponized else 0.0
    hit_v = 1.0 - miss_v
    hit_w = 1.0 - miss_w
    tnr = 1.0 - fpr
    return {
        "threshold": threshold,
        "fpr": fpr,
        "miss_violence": miss_v,
        "miss_weaponized": miss_w,
        "recall_violence": hit_v,
        "recall_weaponized": hit_w,
        "recall_positive": float(alert[is_violence | is_weaponized].mean()),
        "precision": float(alert[is_violence | is_weaponized].sum() / alert.sum())
        if alert.any()
        else 0.0,
        "alert_rate_test": float(alert.mean()),
    }


def expected_cost_production(r: dict) -> float:
    """Reweight class-conditional error rates with the locked production mix."""
    return (
        PROD_P_NORMAL * r["fpr"] * COST_FP
        + PROD_P_NORMAL * (1.0 - r["fpr"]) * COST_TN
        + PROD_P_VIOLENCE * r["miss_violence"] * COST_FN_VIOLENCE
        + PROD_P_VIOLENCE * (1.0 - r["miss_violence"]) * COST_TP
        + PROD_P_WEAPONIZED * r["miss_weaponized"] * COST_FN_WEAPONIZED
        + PROD_P_WEAPONIZED * (1.0 - r["miss_weaponized"]) * COST_TP
    )


def mean_cost_on_test(threshold: float) -> float:
    """Empirical €/image on the (imbalanced) test set — not production."""
    alert = p_pos >= threshold
    costs = np.zeros(len(records), dtype=float)
    costs[is_normal & alert] = COST_FP
    costs[is_normal & ~alert] = COST_TN
    costs[is_violence & alert] = COST_TP
    costs[is_violence & ~alert] = COST_FN_VIOLENCE
    costs[is_weaponized & alert] = COST_TP
    costs[is_weaponized & ~alert] = COST_FN_WEAPONIZED
    return float(costs.mean())


# Sweep every distinct score plus endpoints (README: take the minimum on the curve)
thresholds = np.unique(np.concatenate([[0.0], p_pos, [1.0 + 1e-12]]))
sweep_rows = []
for t in thresholds:
    r = rates_at_threshold(float(t))
    r["expected_cost_production"] = expected_cost_production(r)
    r["mean_cost_test"] = mean_cost_on_test(float(t))
    r["expected_cost_per_1000_production"] = 1000.0 * r["expected_cost_production"]
    sweep_rows.append(r)

sweep_rows.sort(key=lambda row: row["threshold"])
best = min(sweep_rows, key=lambda row: row["expected_cost_production"])
# first row (t=0) alerts on everything; last row (t>max score) never alerts
always_alert = sweep_rows[0]
always_clear = sweep_rows[-1]

print("\n=== Single-threshold sweep (production-reweighted €/frame) ===")
print(
    f"costs FP={COST_FP} FN_v={COST_FN_VIOLENCE} FN_w={COST_FN_WEAPONIZED} "
    f"TP={COST_TP} TN={COST_TN}"
)
print(
    f"production P(pos)={PROD_P_POSITIVE}  "
    f"P(v)={PROD_P_VIOLENCE}  P(w)={PROD_P_WEAPONIZED}"
)
print(
    f"cost-curve minimum t={best['threshold']:.6f}  "
    f"E[USD/frame]={best['expected_cost_production']:.6g}  "
    f"E[USD/1000]={best['expected_cost_per_1000_production']:.4g}  "
    f"FPR={best['fpr']:.3f} miss_v={best['miss_violence']:.3f} "
    f"miss_w={best['miss_weaponized']:.3f}"
)
print(
    f"always-clear: E[USD/frame]={always_clear['expected_cost_production']:.6g}  "
    f"always-alert: E[USD/frame]={always_alert['expected_cost_production']:.6g}"
)
# Alert at the machine-only cost minimum; clear only very confident calm scores.
T_CLEAR = 0.01
T_ALERT = float(best["threshold"])
print(
    f"shipped operating point: t_clear={T_CLEAR:.6f} t_alert={T_ALERT:.6f} "
    "(wide review band between cutoffs)"
)


In [ ]:
# Histogram + ECDFs of p_positive by true class (test set)
bins = np.linspace(0, 1, 41)
colors = {"normal": "#4a4a4a", "violence": "#c45c26", "weaponized": "#1f4e79"}

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

ax = axes[0]
for label in ("normal", "violence", "weaponized"):
    scores = p_pos[y_true == label]
    ax.hist(
        scores,
        bins=bins,
        density=True,
        histtype="step",
        lw=1.8,
        color=colors[label],
        label=f"{label} (n={len(scores)})",
    )
ax.set_xlabel("p_positive")
ax.set_ylabel("density")
ax.set_title("Score distribution by true class")
ax.set_xlim(0, 1)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

ax = axes[1]
for label in ("normal", "violence", "weaponized"):
    scores = np.sort(p_pos[y_true == label])
    ecdf = np.arange(1, len(scores) + 1) / len(scores)
    ax.plot(scores, ecdf, color=colors[label], lw=1.8, label=label)
ax.set_xlabel("p_positive")
ax.set_ylabel("fraction of class ≤ score")
ax.set_title("ECDF by true class")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

if T_CLEAR is not None:
    for ax in axes:
        ax.axvline(T_CLEAR, color="#2a9d8f", ls="--", lw=1.0, label="t_clear")
if T_ALERT is not None:
    for ax in axes:
        ax.axvline(T_ALERT, color="#e76f51", ls="--", lw=1.0, label="t_alert")

fig.tight_layout()
dist_path = ARTIFACTS_DIR / "score_distribution.png"
fig.savefig(dist_path, dpi=120)
plt.show()
print("wrote", dist_path)

# Percentiles — useful anchors when choosing the band
print("\np_positive percentiles by true class:")
print(f"{'class':12s} {'p10':>7s} {'p25':>7s} {'p50':>7s} {'p75':>7s} {'p90':>7s}")
for label in ("normal", "violence", "weaponized"):
    q = np.percentile(p_pos[y_true == label], [10, 25, 50, 75, 90])
    print(f"{label:12s} {q[0]:7.3f} {q[1]:7.3f} {q[2]:7.3f} {q[3]:7.3f} {q[4]:7.3f}")


In [ ]:
# Cost-against-threshold curve (production € per 1,000 frames)
ts = [row["threshold"] for row in sweep_rows if row["threshold"] <= 1.0]
costs_1k = [
    row["expected_cost_per_1000_production"]
    for row in sweep_rows
    if row["threshold"] <= 1.0
]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(ts, costs_1k, color="#1f4e79", lw=1.5, label="expected € / 1,000 frames")
ax.axvline(best["threshold"], color="#c45c26", ls="--", lw=1.2, label=f"argmin t={best['threshold']:.3f}")
ax.axhline(
    always_clear["expected_cost_per_1000_production"],
    color="#666666",
    ls=":",
    lw=1.0,
    label="always-clear",
)
if T_CLEAR is not None:
    ax.axvline(T_CLEAR, color="#2a9d8f", ls="-", lw=1.0, label=f"t_clear={T_CLEAR}")
if T_ALERT is not None:
    ax.axvline(T_ALERT, color="#e76f51", ls="-", lw=1.0, label=f"t_alert={T_ALERT}")
ax.set_xlabel("threshold on p_positive (alert if score ≥ t)")
ax.set_ylabel("expected € per 1,000 frames (production mix)")
ax.set_title("Cost vs threshold — complementary 1 fps CCTV")
ax.set_xlim(0, 1)
ax.legend(loc="best", fontsize=8)
ax.grid(True, alpha=0.3)
fig.tight_layout()

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
curve_path = ARTIFACTS_DIR / "cost_vs_threshold.png"
fig.savefig(curve_path, dpi=120)
plt.show()
print("wrote", curve_path)


In [ ]:
def band_outcomes(t_clear: float, t_alert: float) -> dict:
    """Three-way policy on p_positive. Human review assumed perfect if COST_REVIEW set."""
    if not (0.0 <= t_clear <= t_alert <= 1.0 + 1e-9):
        raise ValueError(f"need 0 <= t_clear <= t_alert <= 1, got {t_clear}, {t_alert}")
    clear = p_pos < t_clear
    alert = p_pos >= t_alert
    review = ~(clear | alert)

    auto_fpr = float(alert[is_normal].mean()) if n_normal else 0.0
    auto_miss_v = float(clear[is_violence].mean()) if n_violence else 0.0
    auto_miss_w = float(clear[is_weaponized].mean()) if n_weaponized else 0.0
    review_given = {
        "normal": float(review[is_normal].mean()) if n_normal else 0.0,
        "violence": float(review[is_violence].mean()) if n_violence else 0.0,
        "weaponized": float(review[is_weaponized].mean()) if n_weaponized else 0.0,
    }
    review_share_test = float(review.mean())
    review_share_prod = (
        PROD_P_NORMAL * review_given["normal"]
        + PROD_P_VIOLENCE * review_given["violence"]
        + PROD_P_WEAPONIZED * review_given["weaponized"]
    )

    if COST_REVIEW is None:
        e_cost = None
    else:
        # Perfect reviewer: middle band pays COST_REVIEW and causes no FP/FN.
        e_cost = (
            PROD_P_NORMAL * auto_fpr * COST_FP
            + PROD_P_VIOLENCE * auto_miss_v * COST_FN_VIOLENCE
            + PROD_P_WEAPONIZED * auto_miss_w * COST_FN_WEAPONIZED
            + review_share_prod * COST_REVIEW
        )

    return {
        "t_clear": t_clear,
        "t_alert": t_alert,
        "review_share_test": review_share_test,
        "review_share_production": review_share_prod,
        "review_given_class": review_given,
        "auto_alert_share_test": float(alert.mean()),
        "auto_clear_share_test": float(clear.mean()),
        "auto_fpr": auto_fpr,
        "auto_miss_violence": auto_miss_v,
        "auto_miss_weaponized": auto_miss_w,
        "expected_cost_production": e_cost,
        "expected_cost_per_1000_production": None if e_cost is None else 1000.0 * e_cost,
    }


# Coarse grid so you can see review load before locking cutoffs
grid = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
print("t_clear  t_alert  review%_test  review%_prod  auto_FPR  miss_v  miss_w")
band_grid = []
for tc in grid:
    for ta in grid:
        if ta < tc:
            continue
        row = band_outcomes(tc, ta)
        band_grid.append(row)
        if tc in (0.1, 0.2, 0.3) and ta in (0.7, 0.8, 0.9):
            print(
                f"{tc:6.1f}  {ta:6.1f}  {100*row['review_share_test']:8.1f}%  "
                f"{100*row['review_share_production']:8.3f}%  "
                f"{row['auto_fpr']:7.3f}  {row['auto_miss_violence']:5.3f}  "
                f"{row['auto_miss_weaponized']:5.3f}"
            )

chosen_band = None
if T_CLEAR is not None and T_ALERT is not None:
    chosen_band = band_outcomes(float(T_CLEAR), float(T_ALERT))
    print("\n=== Your operating band ===")
    for k, v in chosen_band.items():
        print(f"  {k}: {v}")
else:
    print("\nT_CLEAR / T_ALERT still None — set them in the cell above after you pick.")


In [ ]:
# Persist sweep + (possibly unset) operating point for app.py / REPORT.md
cost_payload = {
    "costs_usd": {
        "fp": COST_FP,
        "fn_violence": COST_FN_VIOLENCE,
        "fn_weaponized": COST_FN_WEAPONIZED,
        "tp": COST_TP,
        "tn": COST_TN,
        "review": COST_REVIEW,
    },
    "production": {
        "sampling": "1 still frame every 30 seconds from CCTV (no video model)",
        "p_positive": PROD_P_POSITIVE,
        "p_weaponized_given_positive": PROD_P_WEAPONIZED_GIVEN_POS,
        "p_violence": PROD_P_VIOLENCE,
        "p_weaponized": PROD_P_WEAPONIZED,
        "p_normal": PROD_P_NORMAL,
    },
    "score": "p_positive = p_violence + p_weaponized",
    "n_test": len(records),
    "single_threshold_sweep": sweep_rows,
    "argmin_production": best,
    "baselines_production": {
        "always_alert": always_alert,
        "always_clear": always_clear,
    },
    "operating_point": {
        "t_clear": T_CLEAR,
        "t_alert": T_ALERT,
        "cost_review": COST_REVIEW,
        "band": chosen_band,
        "status": "unset"
        if T_CLEAR is None or T_ALERT is None
        else "set",
        "note": "t_clear=0.01, t_alert=production-cost argmin; middle band to human review.",
    },
    "artifacts": {
        "curve_png": str((ARTIFACTS_DIR / "cost_vs_threshold.png").relative_to(ROOT)),
        "predictions": str(PRED_PATH.relative_to(ROOT)),
    },
}

out_path = ARTIFACTS_DIR / "cost_sweep.json"
out_path.write_text(json.dumps(cost_payload, indent=2) + "\n", encoding="utf-8")
print("wrote", out_path)
print("operating_point status:", cost_payload["operating_point"]["status"])
